A jupyter notebook to interpolate data to z=300m for visulations.
This script computes this for all output times,
not just the end (or another time).

In [13]:
import numpy as np
from netCDF4 import Dataset
from matplotlib import pyplot as plt
import matplotlib
import matplotlib.colors as colors

In [14]:
def z_interp(h, field_vals_all, lon, lat, z_val):
    field_vals = np.zeros((len(lat), len(lon)))
    for i in np.arange(len(lat)):
        for j in np.arange(len(lon)):
            if h[-1,i,j] > z_val:
                # This value is inside the topography
                field_vals[i,j] = np.nan
            else:
                # Find indices either side of this value
                low_idx = np.where(h[:,i,j] < z_val)[0][0]
                high_idx = np.where(h[:,i,j] > z_val)[0][-1]

                # Compute weightings
                weight_low = (z_val - h[low_idx,i,j])/(h[high_idx,i,j] - h[low_idx,i,j])
                weight_high = 1. - weight_low

                # Compute and store value
                field_vals[i,j] = weight_low*field_vals_all[low_idx, i, j] + weight_high*field_vals_all[high_idx, i, j]
    return field_vals

def cubic_z_interp(h, field_vals_all, lon, lat, z_val):
    # Now make cubic interpolation coefficients for each grid staggering

    # Use the bottom four levels (in Python notation!)
    levels = [-1, -2, -3, -4]
    
    coeffs = np.zeros((4, len(lat), len(lon)))

    # Compute weights using interpolating polynomials
    coeffs[0] = (
        (z_val - h[levels[1]]) * (z_val - h[levels[2]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[0]] - h[levels[1]]) * (h[levels[0]] - h[levels[2]])
        * (h[levels[0]] - h[levels[3]])
    )
    coeffs[1] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[2]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[1]] - h[levels[0]]) * (h[levels[1]] - h[levels[2]])
        * (h[levels[1]] - h[levels[3]])
    )
    coeffs[2] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[1]])
        * (z_val - h[levels[3]])
    ) / (
        (h[levels[2]] - h[levels[0]]) * (h[levels[2]] - h[levels[1]])
        * (h[levels[2]] - h[levels[3]])
    )
    coeffs[3] = (
        (z_val - h[levels[0]]) * (z_val - h[levels[1]])
        * (z_val - h[levels[2]])
    ) / (
        (h[levels[3]] - h[levels[0]]) * (h[levels[3]] - h[levels[1]])
        * (h[levels[3]] - h[levels[2]])
    )

    field_vals = np.zeros((len(lat), len(lon)))
    
    for i in np.arange(4):
        field_vals += coeffs[i]*field_vals_all[levels[i]]

    # Set values below surface to NaN
    field_vals = np.where(
        h[levels[0]] > z_val, np.nan, field_vals
    )
    
    return field_vals

In [15]:
# Choose the data to regrid
#test = 'gap'
test='vortex'

rot = True

#dycore = 'CAM-SE'
#dycore = 'CAM-FV3'
dycore = 'CAM-MPAS'

# Give an extra name to the file, for diffusion testing

# SE
#extra_name = '_diff_2x_weaker'

# Fv3
#extra_name = '_hord5'

# MPAS
#extra_name = '_divfact_1'

# No diffusion
extra_name = ''


In [16]:
if rot:
    rot_state='with_rot'
else:
    rot_state='omega0'

if dycore == 'CAM-SE':
    case = f'cam_6_4_100_se_ne60_ztop20km_L57_new_RF'
    if test == 'gap':
        nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau_100s{extra_name}.nc'
    else:
        nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau100s{extra_name}.nc'
elif dycore == 'CAM-FV3':
    case = f'cam_6_4_070_horiz_mount_flow_fv3_C192_ztop20km_L57_new_RF'
    nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau100s{extra_name}.regrid.0.5x0.5.nc'
elif dycore == 'CAM-MPAS':
    case = f'cam_6_4_080_paper_six_mpasa60_L20km_L57_new_RF_w0'
    nc_file = f'{case}.cam.h0i.0001-01-01-00000_{test}_{rot_state}_tau100s{extra_name}.regrid.0.5x0.5.nc'

run_base = "/glade/derecho/scratch/timand/"
run_path = run_base + case + '/run/' + nc_file
nc = Dataset(run_path)

In [17]:
# Save the regridded data. Do so as net cdf
savename = f'{dycore}_{test}_{rot_state}{extra_name}_over_time'
output_file = Dataset(f'/glade/u/home/timand/dcmip2025_gap_and_vortex/interpolate_data/interp_data/{savename}.nc', 'w')

time = nc['time'][:]
lat = nc['lat'][:] 
lon = nc['lon'][:]

output_file.createDimension('lon', len(lon))
output_file.createDimension('lat', len(lat))
output_file.createDimension('time', len(time))

lon_var = output_file.createVariable('lon', 'f4', ('lon',))
lat_var = output_file.createVariable('lat', 'f4', ('lat',))

output_file.variables['lon'][:] = lon
output_file.variables['lat'][:] = lat

time_var = output_file.createVariable('time', 'f4', ('time',))
output_file.variables['time'][:] = time

In [18]:
# Perform regridding
z_val = 300

output_file.createVariable('U', 'f4', ('time', 'lat', 'lon'))
output_file.createVariable('V', 'f4', ('time', 'lat', 'lon'))
output_file.createVariable('T', 'f4', ('time', 'lat', 'lon'))

print(np.shape(output_file.variables['T']))

for t_idx in np.arange(len(time)):

    print(t_idx)

    U_field_vals = cubic_z_interp(nc['Z3'][t_idx, :, :, :], nc['U'][t_idx, :, :, :], lon, lat, z_val)
    output_file.variables['U'][t_idx, :, :] = U_field_vals
    
    V_field_vals = cubic_z_interp(nc['Z3'][t_idx, :, :, :], nc['V'][t_idx, :, :, :], lon, lat, z_val)
    output_file.variables['V'][t_idx, :, :] = V_field_vals
    
    T_field_vals = cubic_z_interp(nc['Z3'][t_idx, :, :, :], nc['T'][t_idx, :, :, :], lon, lat, z_val)
    output_file.variables['T'][t_idx, :, :] = T_field_vals

output_file.close()

(81, 361, 720)
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
